In [1]:
#importacao inicial
import pandas as pd
import numpy as np

df_dados = pd.read_excel('../data/raw/Dados Bibliograficos UEM_UZ_ISPQ 2025_2026.xls')
df_dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 34178 entries, 0 to 34177
Data columns (total 45 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Direct               0 non-null      float64       
 1   Ordem                0 non-null      float64       
 2   Ano                  34178 non-null  int64         
 3   NoLido               0 non-null      float64       
 4   candidato_codigo     34178 non-null  int64         
 5   apelido              34177 non-null  str           
 6   nome                 34177 non-null  str           
 7   TipoDoc              34178 non-null  str           
 8   DocIdent             34178 non-null  str           
 9   Sexo                 34177 non-null  str           
 10  EstCivil             34177 non-null  str           
 11  pai_nascimento       34176 non-null  str           
 12  ProvNasc             34176 non-null  str           
 13  ProvRes              34176 non-null  str  

In [2]:
df_dados.rename(columns={'pai_nascimento': 'pais_nascimento'}, inplace=True)

In [3]:
print(df_dados[df_dados['ProvNasc'] == 'Luanda'][['ProvNasc', 'distrito_nascimento', 'pais_nascimento']])

df_dados.loc[df_dados['ProvNasc'] == 'Luanda', ['ProvNasc', 'distrito_nascimento']] = 'Estrangeiro'

df_dados['ProvNasc'].value_counts()

      ProvNasc distrito_nascimento pais_nascimento
12776   Luanda         Estrangeiro          Angola


ProvNasc
Cidade de Maputo       12567
Província de Maputo     5645
Sofala                  4783
Inhambane               2457
Gaza                    2255
Zambezia                2098
Nampula                 1333
Manica                  1238
Tete                     863
Cabo Delgado             437
Niassa                   373
Estrangeiro              127
Name: count, dtype: int64

In [4]:
def corrigir_encoding(texto):
    """Reverte mojibake: recodifica como cp1252 e decodifica como utf-8"""
    if not isinstance(texto, str):
        return texto
    try:
        return texto.encode('cp1252').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        return texto 

colunas_texto = df_dados.select_dtypes(include='object').columns
for col in colunas_texto:
    df_dados[col] = df_dados[col].apply(corrigir_encoding)

C:\Users\belci\AppData\Local\Temp\ipykernel_17884\1720817448.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df_dados.select_dtypes(include='object').columns


In [5]:

for col in colunas_texto:
    suspeitos = df_dados[col].astype(str).str.contains('Ã|‡', na=False, regex=True)
    if suspeitos.any():
        print(f"{col}: {suspeitos.sum()} valores ainda suspeitos")
        print(df_dados.loc[suspeitos, col].unique()[:5])

apelido: 427 valores ainda suspeitos
<ArrowStringArray>
['SEBASTIÃO', 'SABÃO', 'FERRÃO', 'SALOMÃO', 'ROMÃO']
Length: 5, dtype: str
nome: 1561 valores ainda suspeitos
<ArrowStringArray>
[              'FLORENTINA JOÃO',                   'MANUEL JOÃO',
 'SHELTON DA CELIA ESTEVÃO LUIS',                'CELESTINO JOÃO',
                   'AMÉLIA JOÃO']
Length: 5, dtype: str
TipoDoc: 876 valores ainda suspeitos
<ArrowStringArray>
[                             'CARTÃO DE ELEITOR',
                              'CARTA DE CONDUÇÃO',
                                    'TALÃO DE BI',
 'CARTÃO DE IDENTIFICAÇÃO DE REQUERENTE DE ASILO']
Length: 4, dtype: str


In [6]:
#colunas  irrelevantes
colunasRemover = [
    "Direct", "Ordem", "NoLido", "Dispensa", "RevProva",
    "celular", "celularAlternativo", "DataReg", "HoraReg", "OperReg"
]
df_dados.drop(columns=colunasRemover, inplace=True)
df_dados.head()

,Ano,candidato_codigo,apelido,nome,TipoDoc,DocIdent,Sexo,EstCivil,pais_nascimento,ProvNasc,...,ISPQ_Opc1,ISPQ_Cod_Opc2,ISPQ_Opc2,UZ_Cod_Opc1,UZ_Opc1,UZ_Cod_Opc2,UZ_Opc2,Status,nuit,data_Nasc
0,2026,10019,SUARES,SONIA,BILHETE DE IDENTIDADE,1233444,FEMININO,Solteiro(a),Andorra,Estrangeiro,...,Engenharia de Aquacultura,80104.0,Engenharia de Processamento e Controlo de Qual...,35100.0,Administração Pública (Beira) – Diurno - UZ,35110.0,Economia (Beira) - Diurno - UZ,NaN,344553521.0,2011-12-11
1,2026,10039,MUARAPAZ,CLÁUDIO ARMANDO,BILHETE DE IDENTIDADE,030106033801F,MASCULINO,Solteiro(a),Mocambique,Nampula,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,177648922.0,2004-05-07
2,2026,10041,MBALANGO,YUNI ADELAIDE ANÍBAL,BILHETE DE IDENTIDADE,110107852412P,FEMININO,Solteiro(a),Mocambique,Cidade de Maputo,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,131513240.0,2007-10-13
3,2026,10047,CHALE,IBRAIMO OMAR,BILHETE DE IDENTIDADE,070102840008F,MASCULINO,Solteiro(a),Mocambique,Sofala,...,NaN,NaN,NaN,35213.0,Engenharia Mecatrónica (Beira) - Nocturno - UZ,35209.0,Engenharia Eléctrica (Beira) - Nocturno - UZ,NaN,123024631.0,1995-07-05
4,2026,10050,CUVACA,ANTONIO DOMINGOS,BILHETE DE IDENTIDADE,070108883250D,MASCULINO,Solteiro(a),Mocambique,Sofala,...,NaN,NaN,NaN,35210.0,Engenharia Informática (Beira) - Diurno - UZ,35208.0,Engenharia Eléctrica (Beira) - Diurno - UZ,NaN,174122997.0,2004-07-13


In [7]:
colunas_uem = ['UEM_Cod_Opc1', 'UEM_Opc1', 'UEM_Cod_Opc2', 'UEM_Opc2']
df_dados.dropna(subset=colunas_uem, thresh=1, inplace=True)
df_dados.shape

(26798, 35)

In [8]:
lista2 = ["ISPQ_Opc1", "ISPQ_Cod_Opc2", "UZ_Cod_Opc1", "UZ_Opc1", "ISPQ_Opc2", "UZ_Opc2", "Status", "ISPQ_Cod_Opc1", "UZ_Cod_Opc2", "nuit"]
df_dados.drop(columns=lista2, inplace=True)

  


In [9]:
#Copia da base de dados original 
df_analise = df_dados.copy()
#padronizacao de colunas
df_analise['UEM_Cod_Opc2'] = df_analise['UEM_Cod_Opc2'].fillna(0).astype(int)
df_analise['UEM_Opc2'] = df_analise['UEM_Opc2'].fillna('Sem Opcao')
df_analise['UEM_Cod_Opc1'] = df_analise['UEM_Cod_Opc1'].astype(int)
#padronizacao de colunas de texto
colunas_texto = [
    'UEM_Opc1', 'UEM_Opc2', 'EscoPU', 'local_exame',
    'distrito_nascimento', 'distrito_residencia',
    'Sexo', 'EstCivil', 'ProvNasc', 'pais_nascimento'
]
for col in colunas_texto:
    df_analise[col] = df_analise[col].str.strip().str.title()

df_analise.sample(10)

,Ano,candidato_codigo,apelido,nome,TipoDoc,DocIdent,Sexo,EstCivil,pais_nascimento,ProvNasc,...,TipoEst,AnocPU,Classif,distrito_nascimento,distrito_residencia,UEM_Cod_Opc1,UEM_Opc1,UEM_Cod_Opc2,UEM_Opc2,data_Nasc
19416,2026,47086,MACUCULE,EDUARDO BALTAZAR,BILHETE DE IDENTIDADE,100208874613M,Masculino,Solteiro(A),Mocambique,Cidade De Maputo,...,NaN,2025,NaN,Kamaxaquene,Boane,10108,Ensino De Inglês - Diurno - Uem,10126,Tradução Português/Inglês - Diurno - Uem,2006-06-05
11941,2026,34986,KAVELANE,BELEZÁRIA LUCAS,BILHETE DE IDENTIDADE,110106286459j,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,NaN,2023,NaN,Kanyaka,Kanyaka,11008,Biologia E Saúde - Diurno - Uem,11014,Ecologia E Conservação Da Biodiversidade Terre...,2005-06-06
27422,2026,56910,CUMBI,SILVA CARLOS,BILHETE DE IDENTIDADE,110301838240F,Masculino,Solteiro(A),Mocambique,Cidade De Maputo,...,NaN,2018,NaN,Kampfumo,Kamubukwana,10201,Psicologia Das Organizações - Nocturno - Uem,10207,Psicologia Escolar E De Necessidades Educativa...,1983-03-14
15411,2026,36314,EDSON,FLÁVIA,BILHETE DE IDENTIDADE,110306471838Q,Masculino,Solteiro(A),Mocambique,Cidade De Maputo,...,NaN,2024,NaN,Kamubukwana,Kamavota,10104,Ciência Política - Diurno - Uem,10100,Administração Pública - Diurno - Uem,2003-12-10
16775,2026,43337,COANA,BENEDITA ARMANDO,BILHETE DE IDENTIDADE,100108889090D,Feminino,Solteiro(A),Mocambique,Província De Maputo,...,NaN,2025,NaN,Matola Cidade,Cidade De Tete,10600,Direito - Diurno - Uem,10100,Administração Pública - Diurno - Uem,2006-03-27
15105,2026,39448,NHANTUMBO,WINA ARLINDO,BILHETE DE IDENTIDADE,100107493024,Feminino,Solteiro(A),Mocambique,Província De Maputo,...,NaN,2023,NaN,Matola Cidade,Kampfumo,10574,Arquivística (Mat-Ii E Por-Ii) - Diurno - Uem,10576,Biblioteconomia (Mat-Ii E Por-Ii)- Diurno - Uem,2006-08-14
15173,2026,38475,PUNGUANE,FELIZARDA ANTÓNIO,BILHETE DE IDENTIDADE,100107320237A,Feminino,Solteiro(A),Mocambique,Província De Maputo,...,NaN,2022,NaN,Matola Cidade,Matola Cidade,10214,Organização E Gestão Da Educação - Diurno - Uem,10502,Marketing E Relações Públicas - Diurno - Uem,2005-09-23
28630,2026,33500,HUMBANE,CIARA MARCELINA JAMES,BILHETE DE IDENTIDADE,110102500636Q,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,NaN,2024,NaN,Kampfumo,Kampfumo,11104,Engenharia Eléctrica - Diurno - Uem,11110,Engenharia Informática - Diurno - Uem,2007-02-10
22218,2026,50824,ASSANE,MALAIKA,BILHETE DE IDENTIDADE,100107722427Q,Feminino,Solteiro(A),Mocambique,Província De Maputo,...,NaN,2025,NaN,Matola Cidade,Matola Cidade,11110,Engenharia Informática - Diurno - Uem,11118,Engenharia De Telecomunicações - Diurno - Uem,2006-09-27
13201,2026,32454,SAMBO,ARSÉNIO MAURÍCIO,BILHETE DE IDENTIDADE,110102487757C,Masculino,Solteiro(A),Mocambique,Cidade De Maputo,...,NaN,2022,NaN,Kanyaka,Kanyaka,10302,Contabilidade E Finanças - Diurno - Uem,11000,Informática (Mat-Ii E Por-Ii) - Diurno - Uem,2004-06-17


In [10]:
lista3 = ['TipoEst', 'Classif']
df_analise.drop(columns=lista3, inplace=True)
display(df_analise.sample(10))
display(df_analise.info())

,Ano,candidato_codigo,apelido,nome,TipoDoc,DocIdent,Sexo,EstCivil,pais_nascimento,ProvNasc,...,EscoPU,local_exame,AnocPU,distrito_nascimento,distrito_residencia,UEM_Cod_Opc1,UEM_Opc1,UEM_Cod_Opc2,UEM_Opc2,data_Nasc
4080,2026,13545,OBJANE,JOCELINE GABRIELLE,BILHETE DE IDENTIDADE,110505271831A,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,Instituto Comercial De Maputo,Cidade De Maputo,2024,Kamubukwana,Kamubukwana,10302,Contabilidade E Finanças - Diurno - Uem,10305,Gestão - Nocturno - Uem,2006-07-07
7883,2026,27268,ANTONIO,ZECA XAVIER,BILHETE DE IDENTIDADE,110105200210B,Masculino,Solteiro(A),Mocambique,Zambezia,...,Outra - Província De Maputo,Cidade De Maputo,2024,Nicodala,Matola Cidade,10600,Direito - Diurno - Uem,11676,Ensino De Filosofia (Por-I E His-I) - Diurno -...,2005-07-08
18264,2026,45581,COSSA,KAYLA FAUSTINO,BILHETE DE IDENTIDADE,110505203500Q,Feminino,Solteiro(A),Africa Do Sul,Estrangeiro,...,Externato Cantinho Do Céu,Província De Maputo,2025,Estrangeiro,Matola Cidade,11110,Engenharia Informática - Diurno - Uem,11000,Informática (Mat-Ii E Por-Ii) - Diurno - Uem,2008-06-02
8423,2026,28310,JUNIOR,BASILIO GONCALVES,BILHETE DE IDENTIDADE,020105545875F,Masculino,Solteiro(A),Mocambique,Cabo Delgado,...,Outra - Província De Nampula,Cidade De Pemba,2021,Pemba Cidade,Pemba Cidade,10700,Engenharia Agronómica - Diurno - Uem,10702,Engenharia Florestal - Diurno - Uem,2006-12-17
11491,2026,33112,MACAMO,MANUELA BERNARDO,BILHETE DE IDENTIDADE,110108930052N,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,Escola Secundária Joaquim Alberto Chissano,Cidade De Maputo,2025,Kamaxaquene,Kamavota,11104,Engenharia Eléctrica - Diurno - Uem,11112,Engenharia Do Ambiente - Diurno - Uem,2011-11-21
29606,2026,59388,FONDO,YUSSI INÁCIO,BILHETE DE IDENTIDADE,100108874327M,Masculino,Solteiro(A),Mocambique,Província De Maputo,...,Outra - Província De Maputo,Província De Maputo,2025,Matola Cidade,Matola Cidade,10600,Direito - Diurno - Uem,0,Sem Opcao,2008-02-08
27208,2026,39674,CARLOS,FRENK FRANCISCO,BILHETE DE IDENTIDADE,100100453366F,Masculino,Solteiro(A),Mocambique,Província De Maputo,...,Escola Secundária Da Matola,Cidade De Maputo,2013,Matola Cidade,Matola Cidade,10800,Medicina - Diurno - Uem,0,Sem Opcao,1995-12-04
30564,2026,60066,CASTRO,ILDA,BILHETE DE IDENTIDADE,110105227881c,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,Escola Secundária Joaquim Alberto Chissano,Província De Maputo,2023,Kamavota,Kamavota,10505,Arquivística (Por-I E His-I) - Nocturno - Uem,10215,Organização E Gestão Da Educação - Nocturno - Uem,2003-12-11
33822,2026,64844,MANUEL,SHANIA MERY,BILHETE DE IDENTIDADE,110108989754D,Feminino,Solteiro(A),Mocambique,Província De Maputo,...,Escola Secundária De Mucoque,Província De Maputo,2023,Matola Cidade,Matola Cidade,10206,Psicologia Escolar E De Necessidades Educativa...,10200,Psicologia Das Organizações- Diurno - Uem,2002-09-06
24932,2026,52653,MUNGUAMBE,DENIO JOSSIAS,BILHETE DE IDENTIDADE,1101088868734P,Masculino,Solteiro(A),Mocambique,Cidade De Maputo,...,Desconhecida - Cidade De Maputo,Cidade De Maputo,2023,Kamavota,Kamavota,11034,Hidrogeologia E Recursos Hídricos - Diurno - Uem,11118,Engenharia De Telecomunicações - Diurno - Uem,2005-11-24


<class 'pandas.DataFrame'>
Index: 26798 entries, 0 to 34177
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Ano                  26798 non-null  int64         
 1   candidato_codigo     26798 non-null  int64         
 2   apelido              26797 non-null  str           
 3   nome                 26797 non-null  str           
 4   TipoDoc              26798 non-null  str           
 5   DocIdent             26798 non-null  str           
 6   Sexo                 26797 non-null  str           
 7   EstCivil             26797 non-null  str           
 8   pais_nascimento      26797 non-null  str           
 9   ProvNasc             26797 non-null  str           
 10  ProvRes              26797 non-null  str           
 11  ProvCand             26798 non-null  str           
 12  cod_preUni           26769 non-null  float64       
 13  EscoPU               26798 non-null  str       

None

In [11]:
#verificar a onformidade dos dados, para evitar problemas de integridade na base de dados
agrupamento = df_analise.groupby(['TipoDoc']).size()
agrupamento

TipoDoc
BI-VITALÍCIO                                        130
BILHETE DE IDENTIDADE                             25592
Bilhete de Identidade                                 9
CARTA DE CONDUÇÃO                                    67
CARTÃO DE ELEITOR                                   159
CARTÃO DE IDENTIFICAÇÃO DE REQUERENTE DE ASILO       17
Carta de Condução                                     1
DIRE                                                  9
PASSAPORTE                                          404
TALÃO DE BI                                         410
dtype: int64

In [14]:
df_analise.loc[df_analise['TipoDoc'] == 'Bilhete de Identidade', 'TipoDoc'] = 'BILHETE DE IDENTIDADE'
agrupamento = df_analise.groupby(['TipoDoc']).size()
agrupamento

TipoDoc
BI-VITALÍCIO                                        130
BILHETE DE IDENTIDADE                             25601
CARTA DE CONDUÇÃO                                    67
CARTÃO DE ELEITOR                                   159
CARTÃO DE IDENTIFICAÇÃO DE REQUERENTE DE ASILO       17
Carta de Condução                                     1
DIRE                                                  9
PASSAPORTE                                          404
TALÃO DE BI                                         410
dtype: int64

In [ ]:
df_analise['idade'] = 2026 - df_analise['data_Nasc'].dt.year.astype('Int64')

display(df_analise.head(2))
df_analise['idade'].describe()

,Ano,candidato_codigo,apelido,nome,TipoDoc,DocIdent,Sexo,EstCivil,pais_nascimento,ProvNasc,...,local_exame,AnocPU,distrito_nascimento,distrito_residencia,UEM_Cod_Opc1,UEM_Opc1,UEM_Cod_Opc2,UEM_Opc2,data_Nasc,idade
0,2026,10019,SUARES,SONIA,BILHETE DE IDENTIDADE,1233444,Feminino,Solteiro(A),Andorra,Estrangeiro,...,Chibuto,2012,Estrangeiro,Pemba Cidade,10100,Administração Pública - Diurno - Uem,10128,Arqueologia E Gestão Do Património Cultural - ...,2011-12-11,15
1,2026,10039,MUARAPAZ,CLÁUDIO ARMANDO,BILHETE DE IDENTIDADE,030106033801F,Masculino,Solteiro(A),Mocambique,Nampula,...,Cidade De Maputo,2022,Cidade De Nampula,Kamaxaquene,10300,Economia - Diurno - Uem,10302,Contabilidade E Finanças - Diurno - Uem,2004-05-07,22


count      26797.0
mean     20.807255
std       4.076333
min           15.0
25%           18.0
50%           20.0
75%           22.0
max           64.0
Name: idade, dtype: Float64

In [ ]:
agrupamento = df_analise.groupby(['Sexo']).size()
agrupamento

Sexo
Feminino     15580
Masculino    11217
dtype: int64

In [ ]:
agrupamento = df_analise.groupby(['ProvNasc']).size()
agrupamento

ProvNasc
Cabo Delgado             357
Cidade De Maputo       12404
Estrangeiro              119
Gaza                    1991
Inhambane               1954
Manica                   541
Nampula                 1046
Niassa                   286
Província De Maputo     5548
Sofala                   985
Tete                     317
Zambezia                1249
dtype: int64

In [ ]:
agrupamento = df_analise.groupby(['pais_nascimento']).size()
agrupamento

pais_nascimento
Africa Do Sul                          78
Alemanha                                1
Andorra                                 1
Angola                                  1
Brasil                                  2
Burundi                                 8
Congo (Republica Democratica Do)        1
Desconhecida                            1
Estrangeiro                             2
Inglaterra                              4
Malawi                                  1
Mocambique                          26678
Nao Indicada                            1
Portugal                                2
Quenia                                  1
Ruanda                                  6
Russia                                  1
Suazilandia                             8
dtype: int64

In [ ]:
df_analise.isnull().sum()

Ano                     0
candidato_codigo        0
apelido                 1
nome                    1
TipoDoc                 0
DocIdent                0
Sexo                    1
EstCivil                1
pais_nascimento         1
ProvNasc                1
ProvRes                 1
ProvCand                0
cod_preUni             29
EscoPU                  0
local_exame             0
AnocPU                  0
distrito_nascimento     1
distrito_residencia     1
UEM_Cod_Opc1            0
UEM_Opc1                0
UEM_Cod_Opc2            0
UEM_Opc2                0
data_Nasc               1
idade                   1
dtype: int64

In [ ]:
# Confirmar a linha nula
linhas_nulas = df_analise[df_analise['nome'].isnull() | df_analise['Sexo'].isnull()]
print(linhas_nulas)

        Ano  candidato_codigo apelido nome                TipoDoc  \
12395  2026             35837     NaN  NaN  BILHETE DE IDENTIDADE   

            DocIdent Sexo EstCivil pais_nascimento ProvNasc  ...  \
12395  110100780169l  NaN      NaN             NaN      NaN  ...   

            local_exame AnocPU  distrito_nascimento distrito_residencia  \
12395  Cidade De Maputo   2025                  NaN                 NaN   

      UEM_Cod_Opc1                 UEM_Opc1 UEM_Cod_Opc2  \
12395        10800  Medicina - Diurno - Uem        10200   

                                        UEM_Opc2  data_Nasc idade  
12395  Psicologia Das Organizações- Diurno - Uem        NaT  <NA>  

[1 rows x 24 columns]


In [ ]:
df_analise = df_analise.dropna(subset=['nome', 'Sexo', 'idade'])
df_analise.isnull().sum()

Ano                     0
candidato_codigo        0
apelido                 0
nome                    0
TipoDoc                 0
DocIdent                0
Sexo                    0
EstCivil                0
pais_nascimento         0
ProvNasc                0
ProvRes                 0
ProvCand                0
cod_preUni             29
EscoPU                  0
local_exame             0
AnocPU                  0
distrito_nascimento     0
distrito_residencia     0
UEM_Cod_Opc1            0
UEM_Opc1                0
UEM_Cod_Opc2            0
UEM_Opc2                0
data_Nasc               0
idade                   0
dtype: int64

In [ ]:

df_analise[df_analise['cod_preUni'].isnull()][['EscoPU', 'ProvCand', 'pais_nascimento']]

,EscoPU,ProvCand,pais_nascimento
1606,Escola Secundária Ndambine 2000,Gaza,Mocambique
2921,Escola Secundária Ndambine 2000,Gaza,Mocambique
4252,Escola Secundária Ndambine 2000,Gaza,Mocambique
5926,Escola Secundária Ndambine 2000,Gaza,Mocambique
6468,Escola Secundária Ndambine 2000,Gaza,Mocambique
7564,Escola Secundária Ndambine 2000,Gaza,Mocambique
7624,Escola Secundária Ndambine 2000,Gaza,Mocambique
9202,Escola Secundária Ndambine 2000,Gaza,Mocambique
10666,Escola Secundária Ndambine 2000,Gaza,Mocambique
11105,Escola Secundária Ndambine 2000,Gaza,Mocambique


In [ ]:

df_analise.loc[
    df_analise['EscoPU'] == 'Escola Secundária Ndambine 2000', 
    'cod_preUni'
] = 'ESC_NDAMBINE_2000'

df_analise.isnull().sum()

In [ ]:
df_analise.to_parquet('../data/processed/candidatos_uem.parquet', index=False)
print("Guardado:", df_analise.shape)

Guardado: (26797, 24)
